# 04 — Ringkasan Hasil

Tabel 3, plot komparatif F1, analisis segmentation failures, feature importance S10.

In [ ]:
# Google Colab Setup
# Jalankan sel ini PERTAMA di setiap session Colab baru.
import os, sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Sesuaikan jika nama folder di Google Drive berbeda
    PROJECT_PATH = '/content/drive/MyDrive/pcd-kelompok-17'
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print('Setup selesai. Project:', PROJECT_PATH)
else:
    print('Lokal - skip mount Drive.')


In [ ]:
# Install dependencies - jalankan sekali per session Colab
%pip install -q opencv-python-headless scipy scikit-image scikit-learn statsmodels matplotlib seaborn pandas tqdm joblib kagglehub tensorflow


In [ ]:
import os
import sys
from pathlib import Path

_in_colab = "COLAB_RELEASE_TAG" in os.environ
ROOT = Path.cwd() if _in_colab else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.evaluate import aggregate_feature_importance, plot_feature_importance, print_summary_table
from src.utils import get_project_paths, read_best_enhancement

paths = get_project_paths()
metrics_dir = paths["metrics"]


## Tabel 3 — Ringkasan Semua Skenario

In [ ]:
summary = print_summary_table(metrics_dir)
summary


## Plot Komparatif F1-Score

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=summary, x="scenario_id", y="f1_weighted", hue="model", ax=ax)
    ax.set_title("F1-Score (weighted) per Skenario")
    ax.set_xlabel("Skenario")
    plt.tight_layout()
    plt.savefig(paths["figures"] / "f1_comparison.png", dpi=150)
    plt.show()


## Enhancement Terpilih (E*)

In [ ]:
print(f"Best enhancement: {read_best_enhancement(metrics_dir)}")


## Segmentation Failures

In [ ]:
fail_path = metrics_dir / "segmentation_failures.csv"
if fail_path.exists():
    fails = pd.read_csv(fail_path)
    display(fails.groupby("commodity").size().sort_values(ascending=False).head(10))
else:
    print("Belum ada log segmentasi. Jalankan skenario dengan segmentasi aktif.")


## Uji Signifikansi McNemar

In [ ]:
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
else:
    print("Jalankan notebook 02 dan 03 untuk uji McNemar.")


## Feature Importance (Skenario 10 — RF)

In [ ]:
import joblib

rf_path = paths["models"] / "rf_scenario_10.pkl"
if rf_path.exists():
    rf = joblib.load(rf_path)
    from src.features import get_feature_group_names
    names = get_feature_group_names("all", segmented=True)
    if len(names) != len(rf.feature_importances_):
        names = [f"f{i}" for i in range(len(rf.feature_importances_))]
    labels, vals = aggregate_feature_importance(rf.feature_importances_, names)
    plot_feature_importance(vals, labels, save_path=paths["figures"] / "feature_importance_s10.png")
    plt.show()
else:
    print("Model RF S10 belum tersedia.")


## Inference Time Comparison

In [ ]:
if not summary.empty and "inference_time_ms_per_image" in summary.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=summary, x="scenario_id", y="inference_time_ms_per_image", ax=ax)
    ax.set_title("Inference Time (ms/image)")
    plt.tight_layout()
    plt.show()
